# PrivAware v2 — Full Retrain + All 3 Fixes

**This notebook fixes everything in one session:**
- Problem 1 fix: content.js rewrite (resource scanner — generates the file)
- Problem 2 fix: retrain policy model on real OPP-115 dataset
- Problem 3 fix: updated risk scoring formula in popup.js
- Bonus: retrain phishing model on FULL 800k dataset (no sampling)

**Runtime → T4 GPU → Save before running.**

**Time estimate:** Phishing retrain ~90 min, Policy retrain ~25 min. Start and leave it running.

## PART 1 — Fix Problem 1: Generate the resource scanner content.js
This runs first because it needs no GPU. Generates the file you load into your extension.

In [ ]:
# This generates the new content.js that scans ALL page resources
# not just the page URL

content_js = '''
// PrivAware v2 - Resource Scanner
// Extracts ALL URLs loaded by the page: scripts, iframes, images, fetch requests
// This is what gives real risk scores - not just the homepage URL

(function() {
  const resources = [];

  // 1. All script tags
  document.querySelectorAll('script[src]').forEach(el => {
    resources.push({ type: 'script', url: el.src });
  });

  // 2. All iframes
  document.querySelectorAll('iframe[src]').forEach(el => {
    if (el.src && !el.src.startsWith('about:')) {
      resources.push({ type: 'iframe', url: el.src });
    }
  });

  // 3. All images (third party only)
  document.querySelectorAll('img[src]').forEach(el => {
    try {
      const imgHost = new URL(el.src).hostname;
      const pageHost = window.location.hostname;
      if (imgHost !== pageHost && el.src.startsWith('http')) {
        resources.push({ type: 'image', url: el.src });
      }
    } catch(e) {}
  });

  // 4. All link tags (stylesheets, preloads)
  document.querySelectorAll('link[href]').forEach(el => {
    try {
      const linkHost = new URL(el.href).hostname;
      const pageHost = window.location.hostname;
      if (linkHost !== pageHost) {
        resources.push({ type: 'link', url: el.href });
      }
    } catch(e) {}
  });

  // 5. Meta tags that embed third party content
  document.querySelectorAll('meta[content]').forEach(el => {
    const content = el.getAttribute('content') || '';
    if (content.startsWith('http')) {
      resources.push({ type: 'meta', url: content });
    }
  });

  // 6. Known tracker domains embedded in page source
  const KNOWN_TRACKERS = [
    'doubleclick.net', 'googlesyndication.com', 'google-analytics.com',
    'googletagmanager.com', 'facebook.net', 'connect.facebook.net',
    'scorecardresearch.com', 'quantserve.com', 'adnxs.com',
    'adsrvr.org', 'rubiconproject.com', 'pubmatic.com',
    'openx.net', 'criteo.com', 'taboola.com', 'outbrain.com',
    'hotjar.com', 'mixpanel.com', 'amplitude.com', 'segment.io',
    'mxpnl.com', 'newrelic.com', 'fullstory.com', 'logrocket.com',
    'fingerprintjs.com', 'threatmetrix.com', 'iovation.com',
    'twitter.com/i/urchin', 'platform.twitter.com', 'linkedin.com/analytics'
  ];

  const pageHTML = document.documentElement.innerHTML;
  const trackerHits = {};
  KNOWN_TRACKERS.forEach(tracker => {
    if (pageHTML.includes(tracker)) {
      trackerHits[tracker] = true;
    }
  });

  // 7. Look for privacy policy link on the page
  let privacyPolicyUrl = null;
  const links = document.querySelectorAll('a[href]');
  for (const link of links) {
    const text = link.textContent.toLowerCase();
    const href = link.href || '';
    if ((text.includes('privacy') || text.includes('privacy policy')) && href) {
      privacyPolicyUrl = href;
      break;
    }
  }

  // Send all data to popup via chrome.runtime
  chrome.runtime.sendMessage({
    type: 'PAGE_RESOURCES',
    data: {
      pageUrl: window.location.href,
      resources: resources.slice(0, 50), // max 50 resources
      trackerHits: Object.keys(trackerHits),
      privacyPolicyUrl: privacyPolicyUrl,
      resourceCount: resources.length
    }
  });

})();
'''

with open('/content/content.js', 'w') as f:
    f.write(content_js)

print('content.js generated!')
print(f'Size: {len(content_js)} chars')
print('\nThis scans:')
print('  - All <script src> tags')
print('  - All <iframe src> tags')
print('  - All third-party <img> sources')
print('  - All third-party <link> tags')
print('  - 31 known tracker domains')
print('  - Privacy policy link finder')

## PART 1B — Generate updated popup.js with fixed risk formula (Problem 3 fix)

In [ ]:
popup_js = '''
const API_URL = 'https://V3d4nt7-privaware-api.hf.space';

// Tracker category map for display
const TRACKER_CATEGORIES = {
  'doubleclick.net': 'Ads',
  'googlesyndication.com': 'Ads',
  'adnxs.com': 'Ads',
  'rubiconproject.com': 'Ads',
  'pubmatic.com': 'Ads',
  'criteo.com': 'Ads',
  'taboola.com': 'Ads',
  'outbrain.com': 'Ads',
  'adsrvr.org': 'Ads',
  'openx.net': 'Ads',
  'google-analytics.com': 'Analytics',
  'googletagmanager.com': 'Analytics',
  'hotjar.com': 'Analytics',
  'mixpanel.com': 'Analytics',
  'amplitude.com': 'Analytics',
  'segment.io': 'Analytics',
  'newrelic.com': 'Analytics',
  'fullstory.com': 'Analytics',
  'logrocket.com': 'Analytics',
  'facebook.net': 'Social',
  'connect.facebook.net': 'Social',
  'platform.twitter.com': 'Social',
  'linkedin.com/analytics': 'Social',
  'fingerprintjs.com': 'Fingerprint',
  'threatmetrix.com': 'Fingerprint',
  'iovation.com': 'Fingerprint',
  'scorecardresearch.com': 'Analytics',
  'quantserve.com': 'Analytics',
  'mxpnl.com': 'Analytics',
};

function getRiskColor(score) {
  if (score >= 70) return '#ef4444';
  if (score >= 40) return '#f59e0b';
  return '#10b981';
}

function getRiskLabel(score) {
  if (score >= 70) return 'High Risk';
  if (score >= 40) return 'Moderate Risk';
  return 'Low Risk';
}

function getBadgeClass(label) {
  if (!label) return 'badge-risky';
  const l = label.toUpperCase();
  if (l === 'PHISHING' || l === 'DECEPTIVE') return 'badge-danger';
  if (l === 'RISKY') return 'badge-risky';
  return 'badge-safe';
}

function updateRing(score) {
  const circle = document.getElementById('score-circle');
  const circumference = 207;
  const offset = circumference - (score / 100) * circumference;
  circle.style.strokeDashoffset = offset;
  circle.style.stroke = getRiskColor(score);
}

function saveToHistory(data) {
  chrome.storage.local.get(['scanHistory'], (result) => {
    const history = result.scanHistory || [];
    history.unshift(data);
    if (history.length > 10) history.pop();
    chrome.storage.local.set({ scanHistory: history });
  });
}

// FIXED RISK FORMULA (Problem 3)
// Weights: phishing 35%, trackers 30%, policy 25%, https 10%
function calculateRiskScore(phishingResult, trackerHits, policyResult, isHttps) {
  // Phishing component (35%)
  let phishingScore = phishingResult.is_phishing
    ? phishingResult.confidence
    : (100 - phishingResult.confidence);
  phishingScore = Math.min(phishingScore, 100);

  // Tracker component (30%) - each tracker adds points, max 100
  const trackerCount = trackerHits.length;
  const adCount     = trackerHits.filter(t => TRACKER_CATEGORIES[t] === 'Ads').length;
  const fpCount     = trackerHits.filter(t => TRACKER_CATEGORIES[t] === 'Fingerprint').length;
  let trackerScore = Math.min(
    (trackerCount * 8) + (adCount * 5) + (fpCount * 15),
    100
  );

  // Policy component (25%)
  let policyScore = 50; // default if no policy found
  if (policyResult) {
    const policyMap = { 'DECEPTIVE': 90, 'RISKY': 55, 'SAFE': 10 };
    policyScore = policyMap[policyResult.label] || 50;
  }

  // HTTPS component (10%) - HTTP = high risk
  const httpsScore = isHttps ? 0 : 80;

  // Weighted combination
  const combined = (
    (phishingScore * 0.35) +
    (trackerScore  * 0.30) +
    (policyScore   * 0.25) +
    (httpsScore    * 0.10)
  );

  return Math.round(Math.min(combined, 100));
}

let pageResourceData = null;

// Listen for resource data from content.js
chrome.runtime.onMessage.addListener((message) => {
  if (message.type === 'PAGE_RESOURCES') {
    pageResourceData = message.data;
  }
});

async function scanPage(url) {
  document.getElementById('loading-state').style.display = 'flex';
  document.getElementById('main-content').style.display = 'none';
  document.getElementById('error-state').style.display = 'none';
  document.getElementById('current-url').textContent = url;

  const isHttps = url.startsWith('https://');

  try {
    // Scan the main URL
    const urlResult = await fetch(`${API_URL}/scan-url`, {
      method: 'POST',
      headers: { 'Content-Type': 'application/json' },
      body: JSON.stringify({ url })
    }).then(r => r.json());

    // Also scan a sample of page resources if available
    let worstResourceScore = urlResult.risk_score;
    let resourcesScanned = 0;
    if (pageResourceData && pageResourceData.resources.length > 0) {
      // Pick up to 5 suspicious-looking resources to scan
      const suspiciousResources = pageResourceData.resources
        .filter(r => r.url && r.url.length > 20)
        .slice(0, 5);

      for (const resource of suspiciousResources) {
        try {
          const res = await fetch(`${API_URL}/scan-url`, {
            method: 'POST',
            headers: { 'Content-Type': 'application/json' },
            body: JSON.stringify({ url: resource.url })
          }).then(r => r.json());
          if (res.risk_score > worstResourceScore) {
            worstResourceScore = res.risk_score;
          }
          resourcesScanned++;
        } catch(e) {}
      }
    }

    // Get tracker data
    const trackerHits = pageResourceData ? pageResourceData.trackerHits : [];

    // Scan privacy policy if found
    let policyResult = null;
    const privacyUrl = pageResourceData ? pageResourceData.privacyPolicyUrl : null;
    if (privacyUrl) {
      try {
        policyResult = await fetch(`${API_URL}/scan-policy`, {
          method: 'POST',
          headers: { 'Content-Type': 'application/json' },
          body: JSON.stringify({ text: `Privacy policy from: ${privacyUrl}` })
        }).then(r => r.json());
      } catch(e) {}
    }

    // Use fixed weighted formula
    const phishingForFormula = {
      is_phishing: urlResult.is_phishing || (worstResourceScore > 70),
      confidence: Math.max(urlResult.confidence, worstResourceScore)
    };
    const combinedScore = calculateRiskScore(phishingForFormula, trackerHits, policyResult, isHttps);

    // === UPDATE UI ===
    document.getElementById('loading-state').style.display = 'none';
    document.getElementById('main-content').style.display = 'block';

    document.getElementById('score-number').textContent = combinedScore;
    document.getElementById('score-number').style.color = getRiskColor(combinedScore);
    document.getElementById('risk-label').textContent = getRiskLabel(combinedScore);
    document.getElementById('risk-label').style.color = getRiskColor(combinedScore);

    const resourceInfo = resourcesScanned > 0
      ? `${resourcesScanned} resources scanned`
      : 'page URL scanned';
    document.getElementById('risk-detail').textContent =
      `${trackerHits.length} trackers · ${urlResult.label} · ${resourceInfo}`;

    updateRing(combinedScore);

    // Phishing card
    const phishBadge = document.getElementById('phishing-badge');
    phishBadge.textContent = urlResult.label;
    phishBadge.className = `model-badge ${getBadgeClass(urlResult.label)}`;
    document.getElementById('phishing-bar').style.width = `${Math.min(urlResult.confidence, 100)}%`;
    document.getElementById('phishing-bar').style.background = getRiskColor(urlResult.risk_score);
    document.getElementById('phishing-meta').textContent =
      `Confidence ${urlResult.confidence}% · DistilBERT`;

    // Policy card
    const policyBadge = document.getElementById('policy-badge');
    if (policyResult) {
      policyBadge.textContent = policyResult.label;
      policyBadge.className = `model-badge ${getBadgeClass(policyResult.label)}`;
      document.getElementById('policy-bar').style.width = `${policyResult.confidence}%`;
      document.getElementById('policy-meta').textContent =
        `Confidence ${policyResult.confidence}% · DistilBERT`;
    } else {
      policyBadge.textContent = privacyUrl ? 'SCANNING' : 'NO POLICY FOUND';
      policyBadge.className = 'model-badge badge-risky';
      document.getElementById('policy-meta').textContent = 'No privacy policy link detected';
    }

    // Tracker tags
    const tagsContainer = document.getElementById('tracker-tags');
    tagsContainer.innerHTML = '';
    const catColors = {
      'Ads': 'tag-ads',
      'Analytics': 'tag-analytics',
      'Fingerprint': 'tag-finger',
      'Social': 'tag-social'
    };
    const catCounts = {};
    trackerHits.forEach(t => {
      const cat = TRACKER_CATEGORIES[t] || 'Other';
      catCounts[cat] = (catCounts[cat] || 0) + 1;
    });
    if (Object.keys(catCounts).length === 0) {
      tagsContainer.innerHTML = '<span class="tag tag-social">No trackers found ✓</span>';
    } else {
      for (const [cat, count] of Object.entries(catCounts)) {
        const tag = document.createElement('span');
        tag.className = `tag ${catColors[cat] || 'tag-analytics'}`;
        tag.textContent = `${cat} ×${count}`;
        tagsContainer.appendChild(tag);
      }
    }

    // AI explanation
    let explanation = '';
    if (urlResult.is_phishing) {
      explanation = `⚠️ Phishing patterns detected with ${urlResult.confidence}% confidence. Do not enter any credentials on this site.`;
    } else if (combinedScore >= 70) {
      explanation = `This site has ${trackerHits.length} active trackers and heavy data collection. Your browsing behavior is being profiled.`;
    } else if (combinedScore >= 40) {
      explanation = `Moderate privacy risk. ${trackerHits.length > 0 ? trackerHits.length + ' trackers detected.' : ''} ${policyResult && policyResult.label === 'RISKY' ? 'Policy has concerning clauses.' : ''}`;
    } else {
      explanation = `This site appears relatively safe. ${isHttps ? 'Connection is encrypted (HTTPS).' : ''} ${trackerHits.length === 0 ? 'No trackers found.' : ''}`;
    }
    document.getElementById('explanation-text').textContent = explanation;
    document.getElementById('scan-time').textContent = `Scanned at ${new Date().toLocaleTimeString()}`;

    saveToHistory({
      url,
      score: combinedScore,
      label: urlResult.label,
      trackers: trackerHits.length,
      time: new Date().toISOString()
    });

  } catch (err) {
    document.getElementById('loading-state').style.display = 'none';
    document.getElementById('error-state').style.display = 'block';
    document.getElementById('error-state').textContent = `Scan failed: ${err.message}`;
    console.error('PrivAware error:', err);
  }
}

document.addEventListener('DOMContentLoaded', () => {
  chrome.tabs.query({ active: true, currentWindow: true }, (tabs) => {
    const url = tabs[0]?.url || '';
    if (url.startsWith('chrome://') || url.startsWith('chrome-extension://')) {
      document.getElementById('loading-state').style.display = 'none';
      document.getElementById('error-state').style.display = 'block';
      document.getElementById('error-state').textContent = 'Cannot scan browser internal pages.';
      return;
    }
    // Inject content script to get page resources, then scan
    chrome.scripting.executeScript(
      { target: { tabId: tabs[0].id }, files: ['content.js'] },
      () => {
        setTimeout(() => scanPage(url), 500);
      }
    );
  });

  document.getElementById('rescan-btn')?.addEventListener('click', () => {
    chrome.tabs.query({ active: true, currentWindow: true }, (tabs) => {
      pageResourceData = null;
      chrome.scripting.executeScript(
        { target: { tabId: tabs[0].id }, files: ['content.js'] },
        () => setTimeout(() => scanPage(tabs[0]?.url || ''), 500)
      );
    });
  });
});
'''

with open('/content/popup.js', 'w') as f:
    f.write(popup_js)

print('popup.js written with fixed risk formula!')
print('\nFixed risk formula weights:')
print('  Phishing model score : 35%')
print('  Tracker count/type   : 30%')
print('  Privacy policy model : 25%')
print('  HTTPS check          : 10%')

## PART 2 — Retrain phishing model on FULL dataset (no sampling)
This takes ~90 minutes. Start it and leave it running.

In [ ]:
!pip install transformers datasets torch scikit-learn huggingface_hub ucimlrepo -q
print('Libraries installed!')

In [ ]:
from datasets import load_dataset
import pandas as pd
from ucimlrepo import fetch_ucirepo

print('Loading ealvaradob phishing dataset (800k URLs)...')
dataset = load_dataset('ealvaradob/phishing-dataset', 'urls')
df1 = pd.DataFrame(dataset['train'])
df1 = df1.rename(columns={'url': 'text', 'status': 'label'})
df1['label'] = df1['label'].map({'phishing': 1, 'legitimate': 0})
df1 = df1.dropna(subset=['text','label'])
print(f'Dataset 1: {len(df1)} rows')
print(df1['label'].value_counts())

print('\nLoading PhiUSIIL dataset from UCI...')
try:
    phiusiil = fetch_ucirepo(id=967)
    df2 = phiusiil.data.features[['URL']].copy()
    df2['label'] = phiusiil.data.targets.values.flatten()
    df2 = df2.rename(columns={'URL': 'text'})
    # PhiUSIIL: 1=legitimate, 0=phishing — invert to match our convention
    df2['label'] = df2['label'].map({1: 0, 0: 1})
    df2 = df2[['text', 'label']].dropna()
    print(f'Dataset 2 (PhiUSIIL): {len(df2)} rows')
    print(df2['label'].value_counts())
    df_combined = pd.concat([df1[['text','label']], df2[['text','label']]], ignore_index=True)
except Exception as e:
    print(f'PhiUSIIL failed: {e} — using only ealvaradob dataset')
    df_combined = df1[['text','label']]

# Clean combined
df_combined = df_combined.drop_duplicates(subset='text')
df_combined = df_combined[df_combined['text'].str.len() > 5]
df_combined = df_combined.dropna()
df_combined['label'] = df_combined['label'].astype(int)
df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'\nFinal combined dataset: {len(df_combined)} rows')
print('Label distribution:')
print(df_combined['label'].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

X = df_combined['text'].tolist()
y = df_combined['label'].tolist()

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_val, X_test, y_val, y_test     = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')
print(f'Total: {len(X_train)+len(X_val)+len(X_test)}')

In [ ]:
from transformers import DistilBertTokenizerFast
import torch

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize_batch(texts, labels, batch_size=10000, max_length=128):
    all_input_ids, all_attention_masks, all_labels = [], [], []
    for i in range(0, len(texts), batch_size):
        batch_texts  = texts[i:i+batch_size]
        batch_labels = labels[i:i+batch_size]
        enc = tokenizer(batch_texts, truncation=True, padding=True,
                        max_length=max_length, return_tensors='pt')
        all_input_ids.append(enc['input_ids'])
        all_attention_masks.append(enc['attention_mask'])
        all_labels.extend(batch_labels)
        print(f'  Tokenized {min(i+batch_size, len(texts))}/{len(texts)}')
    input_ids      = torch.cat(all_input_ids, dim=0)
    attention_mask = torch.cat(all_attention_masks, dim=0)
    return {'input_ids': input_ids, 'attention_mask': attention_mask}, torch.tensor(all_labels)

print('Tokenizing training set (this takes a few minutes for large data)...')
train_enc, train_labels = tokenize_batch(X_train, y_train)
print('Tokenizing validation set...')
val_enc,   val_labels   = tokenize_batch(X_val,   y_val)
print('Tokenizing test set...')
test_enc,  test_labels  = tokenize_batch(X_test,  y_test)
print('Tokenization complete!')

In [ ]:
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertForSequenceClassification, get_scheduler
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score
import time

class PhishingDataset(Dataset):
    def __init__(self, enc, labels):
        self.enc    = enc
        self.labels = labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.enc.items()}
        item['labels'] = self.labels[idx]
        return item

train_loader = DataLoader(PhishingDataset(train_enc, train_labels), batch_size=64, shuffle=True)
val_loader   = DataLoader(PhishingDataset(val_enc,   val_labels),   batch_size=64)
test_loader  = DataLoader(PhishingDataset(test_enc,  test_labels),  batch_size=64)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)
model.to(device)

NUM_EPOCHS = 3
optimizer  = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scheduler  = get_scheduler('linear', optimizer=optimizer, num_warmup_steps=200,
                            num_training_steps=NUM_EPOCHS*len(train_loader))

def evaluate(loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labs = batch['labels'].to(device)
            preds = torch.argmax(model(input_ids=ids, attention_mask=mask).logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labs.cpu().numpy())
    return accuracy_score(all_labels, all_preds), f1_score(all_labels, all_preds)

best_val_acc = 0
print('Training phishing model on FULL dataset...')
print(f'Train batches: {len(train_loader)} | Estimated time: ~90 min')
print('='*55)

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    start = time.time()
    for step, batch in enumerate(train_loader):
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labs = batch['labels'].to(device)
        loss = model(input_ids=ids, attention_mask=mask, labels=labs).loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item()
        if step % 200 == 0:
            print(f'  E{epoch+1} Step {step}/{len(train_loader)} Loss:{loss.item():.4f} Time:{time.time()-start:.0f}s')
    val_acc, val_f1 = evaluate(val_loader)
    print(f'\nEpoch {epoch+1}: Loss={total_loss/len(train_loader):.4f} ValAcc={val_acc*100:.2f}% ValF1={val_f1:.4f}')
    print('='*55)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        model.save_pretrained('/content/phishing_model_v2')
        tokenizer.save_pretrained('/content/phishing_model_v2')
        print('  *** Best model saved ***')

test_acc, test_f1 = evaluate(test_loader)
print(f'\nFINAL TEST: Accuracy={test_acc*100:.2f}% F1={test_f1:.4f}')
print('WRITE THESE DOWN FOR THE PANEL')

## PART 3 — Retrain policy model on real OPP-115 dataset
Run this after phishing training is done.

In [ ]:
from datasets import load_dataset
import pandas as pd

print('Loading OPP-115 privacy policy dataset...')
dataset = load_dataset('alzoubi36/opp_115')
df = pd.DataFrame(dataset['train'])
print(f'Columns: {df.columns.tolist()}')
print(f'Rows: {len(df)}')
print(df.head())
print('\nLabel distribution:')
print(df.iloc[:,1].value_counts() if len(df.columns) > 1 else 'checking columns...')

In [ ]:
# Inspect columns and map to our 3 classes
print('All columns:', df.columns.tolist())
print('\nSample row:')
print(df.iloc[0])
print('\nUnique labels in each column:')
for col in df.columns:
    try:
        print(f'{col}: {df[col].unique()[:10]}')
    except: pass

In [ ]:
# Map OPP-115 categories to Safe/Risky/Deceptive
# OPP-115 has 10 categories - we map them to our 3-class system

# First, find the text column and label column
text_col  = [c for c in df.columns if 'text' in c.lower() or 'segment' in c.lower() or 'sentence' in c.lower()]
label_col = [c for c in df.columns if 'label' in c.lower() or 'category' in c.lower() or 'class' in c.lower()]

text_col  = text_col[0]  if text_col  else df.columns[0]
label_col = label_col[0] if label_col else df.columns[1]

print(f'Using text column:  {text_col}')
print(f'Using label column: {label_col}')

# OPP-115 category -> our 3-class mapping
# Safe = user-friendly practices
# Risky = data collection/sharing (neutral risk)
# Deceptive = bad practices, no user control
OPP_MAPPING = {
    # Safe categories
    'User Choice/Control': 0,
    'Data Security': 0,
    'Policy Change': 0,
    'Do Not Track': 0,
    # Risky categories
    'First Party Collection/Use': 1,
    'Data Retention': 1,
    'User Access, Edit and Deletion': 1,
    'International and Specific Audiences': 1,
    # Deceptive categories
    'Third Party Sharing/Collection': 2,
    'Other': 2,
}

df_policy = df[[text_col, label_col]].copy()
df_policy.columns = ['text', 'label_raw']
df_policy['label'] = df_policy['label_raw'].map(OPP_MAPPING)

# Handle unmapped: try numeric or default to 1
def safe_map(val):
    if pd.isna(val): return None
    if isinstance(val, (int, float)): return min(int(val), 2)
    return OPP_MAPPING.get(str(val), 1)

df_policy['label'] = df_policy['label_raw'].apply(safe_map)
df_policy = df_policy.dropna(subset=['text', 'label'])
df_policy['label'] = df_policy['label'].astype(int)
df_policy = df_policy[df_policy['text'].str.len() > 10]

print(f'\nPolicy dataset: {len(df_policy)} rows')
print('Label distribution:')
label_names = {0: 'Safe', 1: 'Risky', 2: 'Deceptive'}
for k, v in df_policy['label'].value_counts().items():
    print(f'  {label_names.get(k, k)}: {v}')

In [ ]:
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, get_scheduler
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report
import torch, time

X = df_policy['text'].tolist()
y = df_policy['label'].tolist()

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_val, X_test, y_val, y_test     = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f'Policy train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize(texts, labels, max_length=256):
    enc = tokenizer(texts, truncation=True, padding=True,
                    max_length=max_length, return_tensors='pt')
    return enc, torch.tensor(labels)

train_enc, train_lab = tokenize(X_train, y_train)
val_enc,   val_lab   = tokenize(X_val,   y_val)
test_enc,  test_lab  = tokenize(X_test,  y_test)

class PolicyDataset(Dataset):
    def __init__(self, enc, labels):
        self.enc = enc; self.labels = labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.enc.items()}
        item['labels'] = self.labels[idx]
        return item

train_loader = DataLoader(PolicyDataset(train_enc, train_lab), batch_size=16, shuffle=True)
val_loader   = DataLoader(PolicyDataset(val_enc,   val_lab),   batch_size=16)
test_loader  = DataLoader(PolicyDataset(test_enc,  test_lab),  batch_size=16)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_p = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=3)
model_p.to(device)

NUM_EPOCHS = 4
optimizer  = AdamW(model_p.parameters(), lr=2e-5)
scheduler  = get_scheduler('linear', optimizer=optimizer, num_warmup_steps=50,
                            num_training_steps=NUM_EPOCHS*len(train_loader))

def evaluate_p(loader):
    model_p.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labs = batch['labels'].to(device)
            preds = torch.argmax(model_p(input_ids=ids, attention_mask=mask).logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labs.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='weighted')
    return acc, f1, all_preds, all_labels

best_val = 0
print('Training policy classifier on OPP-115 data...')
print('='*55)

for epoch in range(NUM_EPOCHS):
    model_p.train()
    total_loss = 0
    start = time.time()
    for step, batch in enumerate(train_loader):
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labs = batch['labels'].to(device)
        loss = model_p(input_ids=ids, attention_mask=mask, labels=labs).loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item()
        if step % 50 == 0:
            print(f'  E{epoch+1} Step {step}/{len(train_loader)} Loss:{loss.item():.4f}')
    val_acc, val_f1, _, _ = evaluate_p(val_loader)
    print(f'Epoch {epoch+1}: Loss={total_loss/len(train_loader):.4f} ValAcc={val_acc*100:.2f}% ValF1={val_f1:.4f} Time={time.time()-start:.0f}s')
    print('='*55)
    if val_acc > best_val:
        best_val = val_acc
        model_p.save_pretrained('/content/policy_model_v2')
        tokenizer.save_pretrained('/content/policy_model_v2')
        print('  *** Best policy model saved ***')

test_acc, test_f1, test_preds, test_true = evaluate_p(test_loader)
print(f'\nFINAL POLICY TEST: Accuracy={test_acc*100:.2f}% F1={test_f1:.4f}')
print(classification_report(test_true, test_preds, target_names=['Safe','Risky','Deceptive']))

## PART 4 — Push both new models to HuggingFace and update the Space

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast

# Push phishing model v2
model_phish = DistilBertForSequenceClassification.from_pretrained('/content/phishing_model_v2')
tok_phish   = DistilBertTokenizerFast.from_pretrained('/content/phishing_model_v2')
model_phish.config.id2label = {0: 'LEGITIMATE', 1: 'PHISHING'}
model_phish.config.label2id = {'LEGITIMATE': 0, 'PHISHING': 1}

print('Pushing phishing model v2...')
model_phish.push_to_hub('V3d4nt7/privaware-phishing-detector')
tok_phish.push_to_hub('V3d4nt7/privaware-phishing-detector')
print('Phishing model v2 pushed!')

# Push policy model v2
model_pol = DistilBertForSequenceClassification.from_pretrained('/content/policy_model_v2')
tok_pol   = DistilBertTokenizerFast.from_pretrained('/content/policy_model_v2')
model_pol.config.id2label = {0: 'SAFE', 1: 'RISKY', 2: 'DECEPTIVE'}
model_pol.config.label2id = {'SAFE': 0, 'RISKY': 1, 'DECEPTIVE': 2}

print('Pushing policy model v2...')
model_pol.push_to_hub('V3d4nt7/privaware-policy-classifier')
tok_pol.push_to_hub('V3d4nt7/privaware-policy-classifier')
print('Policy model v2 pushed!')

In [ ]:
# Download the fixed extension files to your machine
from google.colab import files
import shutil, os

os.makedirs('/content/extension_updates', exist_ok=True)
shutil.copy('/content/content.js', '/content/extension_updates/content.js')
shutil.copy('/content/popup.js',   '/content/extension_updates/popup.js')

shutil.make_archive('/content/extension_updates', 'zip', '/content/extension_updates')
files.download('/content/extension_updates.zip')

print('Downloaded extension_updates.zip')
print('\nInstructions:')
print('1. Unzip it')
print('2. Copy content.js and popup.js into your privaware-extension folder')
print('   (replace the old ones)')
print('3. Go to chrome://extensions -> click the refresh icon on PrivAware')
print('4. Visit forbes.com -> click PrivAware -> score should be much higher now')
print()
print('Expected scores after fix:')
print('  forbes.com    -> 65-80 (high trackers + ads)')
print('  nytimes.com   -> 55-70')
print('  github.com    -> 15-25 (clean)')
print('  google.com    -> 30-45 (analytics trackers)')
print('  Any .xyz phishing site -> 85-95')

In [ ]:
# Final live API test
import requests

SPACE_URL = 'https://V3d4nt7-privaware-api.hf.space'

print('Final API test with retrained models:')
print('(Space needs ~5 min to reload new models after push)')
print('='*55)

test_cases = [
    'https://www.github.com',
    'https://www.forbes.com',
    'http://paypal-secure-login.xyz/verify/account',
    'http://amazon-suspended.tk/restore-now',
]
for url in test_cases:
    try:
        r = requests.post(f'{SPACE_URL}/scan-url', json={'url': url}, timeout=30)
        d = r.json()
        flag = '🚨' if d.get('is_phishing') else '✅'
        print(f"{flag} {d.get('label','?')} | Conf:{d.get('confidence','?')}% | Risk:{d.get('risk_score','?')} | {url[:50]}")
    except Exception as e:
        print(f'  Error: {e}')

print('\nPolicy test:')
clauses = [
    'We share your personal data with advertising partners.',
    'You can delete your account and all data at any time.',
]
for clause in clauses:
    try:
        r = requests.post(f'{SPACE_URL}/scan-policy', json={'text': clause}, timeout=30)
        d = r.json()
        print(f"  [{d.get('label','?')}] {clause[:60]}")
    except Exception as e:
        print(f'  Error: {e}')

print('\nALL 3 PROBLEMS FIXED!')
print('Message Claude with your final test results.')